In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random, json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')

WEIGHTS_DIR       = '../../weights/cnn'
RESULTS_TABLE_DIR = '../../results/tables'
RESULTS_PLOT_DIR  = '../../results/plots'
INTEL_TEST_DIR    = '../../data/archive-6/seg_test/seg_test'

os.makedirs(RESULTS_TABLE_DIR, exist_ok=True)
os.makedirs(RESULTS_PLOT_DIR,  exist_ok=True)

from src.cnn.evaluate import ModelEvaluator
print('Setup OK')

## 1. Muat Hasil Training (F1 + Loss Histories)

In [ ]:
results_path = os.path.join(WEIGHTS_DIR, 'training_results.json')

with open(results_path) as f:
    cnn_results = json.load(f)

# Flatten semua variant ke satu dict
all_variants = {}
for group, variants in cnn_results.items():
    for name, info in variants.items():
        all_variants[name] = info

# Build histories dict langsung dari training_results.json
cnn_histories = {name: info['history'] for name, info in all_variants.items()}

df_results = pd.DataFrame([
    {'model': k, 'macro_f1': v['macro_f1']}
    for k, v in all_variants.items()
]).sort_values('macro_f1', ascending=False)

display(df_results)
best_model_name = df_results.iloc[0]['model']
print(f'\nArsitektur terbaik: {best_model_name}  (F1={df_results.iloc[0]["macro_f1"]:.4f})')

## 2. Load Test Dataset

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

test_ds = tf.keras.utils.image_dataset_from_directory(
    INTEL_TEST_DIR,
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
test_ds = test_ds.map(lambda x, y: (x / 255.0, y))
NUM_CLASSES = len(test_ds.class_names)
print(f'Classes ({NUM_CLASSES}): {test_ds.class_names}')

## 3. Forward Propagation From Scratch — Arsitektur Terbaik (Conv2D Shared)

In [ ]:
evaluator = ModelEvaluator(WEIGHTS_DIR)

best_result = evaluator.evaluate_model_pair(best_model_name, test_ds)

print(f'\nModel: {best_model_name}')
print(f'  Keras  macro F1 : {best_result["keras_metrics"]["macro_f1"]:.4f}')
print(f'  Scratch macro F1: {best_result["from_scratch_metrics"]["macro_f1"]:.4f}')
print(f'  Predictions match (atol=1e-5): {best_result["predictions_match"]}')
print(f'  Parameter count : {best_result["parameter_count"]:,}')

## 4. Ganti Conv2D → LocallyConnected2D (Non-Shared)

In [ ]:
LC_MODEL_NAME = 'locally_connected2d_best'
lc_path = os.path.join(WEIGHTS_DIR, f'{LC_MODEL_NAME}.h5')

if os.path.exists(lc_path):
    lc_result = evaluator.evaluate_model_pair(LC_MODEL_NAME, test_ds)
    print(f'Model: {LC_MODEL_NAME}')
    print(f'  Keras  macro F1 : {lc_result["keras_metrics"]["macro_f1"]:.4f}')
    print(f'  Scratch macro F1: {lc_result["from_scratch_metrics"]["macro_f1"]:.4f}')
    print(f'  Predictions match: {lc_result["predictions_match"]}')
    print(f'  Parameter count : {lc_result["parameter_count"]:,}')
else:
    print(f'Model {lc_path} tidak ditemukan — pastikan training FASE 1 sudah selesai')
    lc_result = None

## 5. Tabel Perbandingan: Shared vs Non-Shared

In [ ]:
rows = [
    {
        'model':        best_model_name,
        'type':         'Conv2D (shared)',
        'keras_f1':     round(best_result['keras_metrics']['macro_f1'], 4),
        'scratch_f1':   round(best_result['from_scratch_metrics']['macro_f1'], 4),
        'params':       best_result['parameter_count'],
        'preds_match':  best_result['predictions_match'],
    }
]

if lc_result:
    rows.append({
        'model':        LC_MODEL_NAME,
        'type':         'LocallyConnected2D (non-shared)',
        'keras_f1':     round(lc_result['keras_metrics']['macro_f1'], 4),
        'scratch_f1':   round(lc_result['from_scratch_metrics']['macro_f1'], 4),
        'params':       lc_result['parameter_count'],
        'preds_match':  lc_result['predictions_match'],
    })

df_cmp = pd.DataFrame(rows)
display(df_cmp)

with open(os.path.join(RESULTS_TABLE_DIR, 'cnn_shared_vs_nonshared.json'), 'w') as f:
    json.dump(rows, f, indent=2)

## 6. Plot: Training/Val Loss per Hyperparameter

In [ ]:
def plot_loss_group(group_histories: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in group_histories.items():
        label = name.replace('conv2d_', '').replace('locally_connected2d_', 'lc_')
        axes[0].plot(h['loss'],     label=label)
        axes[1].plot(h['val_loss'], label=label)
    for ax, ylabel in zip(axes, ['Train Loss', 'Val Loss']):
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

# Variasi jumlah layer
layer_hist = {k: v for k, v in cnn_histories.items() if '_F1_K1_Pmax' in k}
if layer_hist:
    plot_loss_group(layer_hist, 'CNN Loss — Variasi Jumlah Layer',
                    os.path.join(RESULTS_PLOT_DIR, 'cnn_loss_num_layers.png'))

# Variasi jumlah filter
filter_hist = {k: v for k, v in cnn_histories.items() if 'L2_' in k and '_K1_Pmax' in k}
if filter_hist:
    plot_loss_group(filter_hist, 'CNN Loss — Variasi Jumlah Filter',
                    os.path.join(RESULTS_PLOT_DIR, 'cnn_loss_num_filters.png'))

# Variasi ukuran kernel
kernel_hist = {k: v for k, v in cnn_histories.items() if 'L2_F1_' in k and '_Pmax' in k}
if kernel_hist:
    plot_loss_group(kernel_hist, 'CNN Loss — Variasi Ukuran Filter',
                    os.path.join(RESULTS_PLOT_DIR, 'cnn_loss_filter_size.png'))

# Variasi pooling
pool_hist = {k: v for k, v in cnn_histories.items() if 'L2_F1_K1_' in k}
if pool_hist:
    plot_loss_group(pool_hist, 'CNN Loss — Variasi Jenis Pooling',
                    os.path.join(RESULTS_PLOT_DIR, 'cnn_loss_pooling_type.png'))

# Shared vs Non-shared
lc_hist = {k: v for k, v in cnn_histories.items() if 'locally' in k}
shared_hist = {k: v for k, v in cnn_histories.items() if best_model_name == k}
if shared_hist or lc_hist:
    plot_loss_group({**shared_hist, **lc_hist},
                    'CNN Loss — Conv2D (Shared) vs LocallyConnected2D (Non-shared)',
                    os.path.join(RESULTS_PLOT_DIR, 'cnn_shared_vs_nonshared.png'))

## 7. Bar Chart: Macro F1 Semua Variasi

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
names  = df_results['model'].tolist()
f1s    = df_results['macro_f1'].tolist()
colors = ['#DD8452' if 'locally' in n else '#4C72B0' for n in names]

bars = ax.bar(names, f1s, color=colors)
ax.set_title('Macro F1-Score — Semua Variasi CNN', fontsize=14)
ax.set_ylabel('Macro F1', fontsize=12)
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4C72B0', label='Conv2D (shared)'),
    Patch(facecolor='#DD8452', label='LocallyConnected2D (non-shared)'),
])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOT_DIR, 'cnn_f1_all_variants.png'), dpi=150, bbox_inches='tight')
plt.show()

## Analisis

### Keras vs From Scratch
*(isi setelah eksperimen dijalankan)*

### Pengaruh Jumlah Layer Konvolusi
*(isi setelah eksperimen dijalankan)*

### Pengaruh Banyak Filter
*(isi setelah eksperimen dijalankan)*

### Pengaruh Ukuran Filter
*(isi setelah eksperimen dijalankan)*

### Pengaruh Jenis Pooling
*(isi setelah eksperimen dijalankan)*

### Shared (Conv2D) vs Non-Shared (LocallyConnected2D)
*(isi setelah eksperimen dijalankan)*